# Tag a split on Kaggle

This notebook runs one CARIS step, `caris tag`, on a free Kaggle GPU. Everything else runs on your own machine.

Before running it:

- **Settings:** accelerator *GPU T4 x2*, internet *on*.
- **Input:** a Kaggle dataset holding the `splits/` and `images/` folders that `caris split` wrote into `data/`. Leave the dataset zip out.
- **Code:** set `REPO` below to a URL that Kaggle can clone. For a private repository, put a GitHub token in the URL from a Kaggle secret, or upload the repository as a second dataset and install it from there.

Run it with **Save Version → Save & Run All**, so it carries on with the browser closed. A tag run resumes where it stopped, so if a session ends early, run the notebook again.

In [ ]:
REPO = "https://github.com/<you>/CARIS_v2.git"
DATA = "/kaggle/input/caris-data"  # the dataset with splits/ and images/
EXPERIMENT = "experiments/gemma4-26b__mxbai__standalone.toml"
SPLITS = ["val", "test"]
OLLAMA_VERSION = "0.40.16"  # the version you run locally, so both machines tag alike

In [ ]:
import subprocess
import time
import urllib.request

!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):  # wait until the server answers
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version")
        break
    except OSError:
        time.sleep(1)

In [ ]:
!git clone --depth 1 {REPO} /kaggle/working/caris
!pip install --quiet -e /kaggle/working/caris
%cd /kaggle/working/caris

In [ ]:
import tomllib

with open(EXPERIMENT, "rb") as f:
    model = tomllib.load(f)["tagging"]["model"]
!ollama pull {model}

In [ ]:
import os

os.environ["CARIS_DATA"] = DATA
for split in SPLITS:
    !caris tag {EXPERIMENT} --split {split}

The tag runs are in `/kaggle/working/caris/runs/tags/`. When the version has finished, download them from the notebook's **Output** tab, put them in `runs/tags/` of your local repository, and commit them.